# EV/Dynacord Chatbot — Full Vectorstore Rebuild

**Date:** 2026-05-26

Rebuilds the chatbot's retrieval index from the full Drive corpus of 5,125 PDFs.
Produces both:
1. **Phase 1**: TF-IDF index (drop-in replacement for the deployed `data/vectorstore/`)
2. **Phase 2**: ChromaDB index with OpenAI `text-embedding-3-small` (the dense-embedding upgrade)

Outputs are written to a new subfolder in Drive that Claude will pick up via the Drive MCP and commit to the repo.

## Before you run

1. **Add `OPENAI_API_KEY` as a Colab secret** (left sidebar → 🔑 Secrets → New secret). Required for Phase 2.
2. **Runtime → Change runtime type → CPU is fine.** No GPU needed.
3. **Runtime → Run all** — total time ~30-40 min.

## 0 · Install deps + mount Drive

In [ ]:
!pip install -q PyMuPDF scikit-learn openai chromadb

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
import os, sys, json, hashlib, time, pickle
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

PDF_ROOT = '/content/drive/MyDrive/Electro-Voice_Dynacord Website Document Downloads'
OUTPUT_DIR = f'{PDF_ROOT}/vectorstore_rebuild_2026-05-26'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'PDF source: {PDF_ROOT}')
print(f'Output dir: {OUTPUT_DIR}')
assert os.path.isdir(PDF_ROOT), 'Drive folder not found — check it mounted'

## 1 · Walk the PDF tree (one-time, slow on Drive mount)

Colab's Drive mount is slow on first `os.walk` (FUSE overhead, same as Drive Desktop) but much faster than the local Mac path because Colab has no Spotlight / no other apps competing. Expect ~30-90s.

In [ ]:
t0 = time.time()
pdfs = []
for dp, _, files in os.walk(PDF_ROOT):
    # Skip our own output dir
    if 'vectorstore_rebuild_' in dp:
        continue
    for f in files:
        if f.lower().endswith('.pdf'):
            pdfs.append(os.path.join(dp, f))
pdfs.sort()
print(f'Found {len(pdfs):,} PDFs in {time.time()-t0:.1f}s')

## 2 · Copy PDFs to local Colab disk

Same lesson as on the Mac — extracting directly from FUSE-mounted Drive is slow. Copying to Colab's local disk first then extracting is much faster overall.

In [ ]:
import shutil
LOCAL_PDF_DIR = '/content/pdfs'
os.makedirs(LOCAL_PDF_DIR, exist_ok=True)

t0 = time.time()
copied = 0
for src in pdfs:
    rel = os.path.relpath(src, PDF_ROOT)
    dst = os.path.join(LOCAL_PDF_DIR, rel)
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    copied += 1
    if copied % 500 == 0:
        print(f'  copied {copied}/{len(pdfs)} ({time.time()-t0:.0f}s)')
print(f'Done: {copied} files copied in {(time.time()-t0)/60:.1f}m')

## 3 · Extract + chunk all PDFs (parallel)

In [ ]:
import fitz  # PyMuPDF

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
MIN_CHUNK_SIZE = 50


def chunk_text(text):
    if not text or len(text.strip()) < MIN_CHUNK_SIZE:
        return
    text = text.strip()
    start = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        if end >= len(text):
            c = text[start:]
            if len(c.strip()) >= MIN_CHUNK_SIZE:
                yield c.strip()
            break
        b = text.rfind('\n\n', start, end)
        if b == -1 or b <= start: b = text.rfind('. ', start, end)
        if b == -1 or b <= start: b = text.rfind('\n', start, end)
        if b == -1 or b <= start: b = text.rfind(' ', start, end)
        if b == -1 or b <= start: b = end
        c = text[start:b+1].strip()
        if len(c) >= MIN_CHUNK_SIZE:
            yield c
        start = b + 1 - CHUNK_OVERLAP
        if start <= 0:
            start = b + 1


def derive_meta(pdf_path, root):
    rel = os.path.relpath(pdf_path, root)
    parts = Path(rel).parts
    brand = 'Dynacord' if 'Dynacord' in rel else 'Electro-Voice'
    category = parts[0] if parts else 'General'
    return {
        'brand': brand,
        'category': category,
        'filename': parts[-1],
        'relative_path': rel,
    }


def process_pdf(args):
    pdf, root = args
    try:
        doc = fitz.open(pdf)
        text = '\n\n'.join(p.get_text('text') for p in doc if p.get_text('text').strip())
        doc.close()
    except Exception as e:
        return []
    if not text:
        return []
    m = derive_meta(pdf, root)
    out = []
    for i, c in enumerate(chunk_text(text)):
        cid = hashlib.md5(f"{m['relative_path']}:{i}".encode()).hexdigest()
        out.append({'id': cid, 'text': c, 'metadata': {**m, 'chunk_index': i}})
    return out


local_pdfs = sorted([
    os.path.join(dp, f)
    for dp, _, files in os.walk(LOCAL_PDF_DIR)
    for f in files if f.lower().endswith('.pdf')
])
print(f'Local PDFs to process: {len(local_pdfs):,}')

all_chunks = []
failed = 0
t0 = time.time()

with ProcessPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(process_pdf, (p, LOCAL_PDF_DIR)): p for p in local_pdfs}
    for i, fut in enumerate(as_completed(futures)):
        try:
            chunks = fut.result()
            all_chunks.extend(chunks)
        except Exception:
            failed += 1
        if (i+1) % 200 == 0:
            el = time.time() - t0
            print(f'  {i+1}/{len(local_pdfs)} PDFs ({(i+1)/el:.1f}/s, {len(all_chunks):,} chunks)')

el = time.time() - t0
print(f'\nExtraction done in {el/60:.1f}m')
print(f'  Chunks: {len(all_chunks):,}')
print(f'  Failed: {failed}')

## 4 · Build TF-IDF index (Phase 1 output)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = [c['text'] for c in all_chunks]
metadatas = [c['metadata'] for c in all_chunks]
ids = [c['id'] for c in all_chunks]

print(f'Fitting TF-IDF over {len(texts):,} chunks ...')
t0 = time.time()
vec = TfidfVectorizer(
    max_features=50000, stop_words='english',
    ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_df=0.95,
)
mat = vec.fit_transform(texts)
print(f'  matrix {mat.shape}, nnz={mat.nnz:,}, {time.time()-t0:.1f}s')

TFIDF_OUT = f'{OUTPUT_DIR}/tfidf'
os.makedirs(TFIDF_OUT, exist_ok=True)

with open(f'{TFIDF_OUT}/vectorizer.pkl', 'wb') as f:
    pickle.dump(vec, f)
with open(f'{TFIDF_OUT}/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(mat, f)
with open(f'{TFIDF_OUT}/chunks_meta.json', 'w', encoding='utf-8') as f:
    json.dump({'texts': texts, 'metadatas': metadatas, 'ids': ids}, f, ensure_ascii=False)
with open(f'{TFIDF_OUT}/all_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False)

for n in ('vectorizer.pkl', 'tfidf_matrix.pkl', 'chunks_meta.json', 'all_chunks.json'):
    sz = os.path.getsize(f'{TFIDF_OUT}/{n}') / 1024 / 1024
    print(f'  {n}: {sz:.1f} MB')

## 5 · Build ChromaDB index with OpenAI embeddings (Phase 2 output)

**Cost:** ~$2 for ~100k chunks at text-embedding-3-small pricing ($0.02/M tokens).

If you don't have OPENAI_API_KEY in Colab secrets yet, **stop here, add it, and only run this cell**.

In [ ]:
from openai import OpenAI
import chromadb

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
assert OPENAI_API_KEY, 'Set OPENAI_API_KEY in Colab secrets first'
client = OpenAI(api_key=OPENAI_API_KEY)

CHROMA_OUT = f'{OUTPUT_DIR}/chromadb'
os.makedirs(CHROMA_OUT, exist_ok=True)

chroma = chromadb.PersistentClient(path=CHROMA_OUT)
try:
    chroma.delete_collection('ev_dynacord_docs')
except Exception:
    pass
col = chroma.create_collection('ev_dynacord_docs')

BATCH = 100
total_batches = (len(all_chunks) + BATCH - 1) // BATCH
t0 = time.time()

for bidx in range(0, len(all_chunks), BATCH):
    batch = all_chunks[bidx:bidx+BATCH]
    b_texts = [c['text'] for c in batch]
    b_ids = [c['id'] for c in batch]
    b_metas = [
        {k: (v if isinstance(v, (str, int, float, bool)) else str(v)) for k, v in c['metadata'].items()}
        for c in batch
    ]
    embs = [d.embedding for d in client.embeddings.create(input=b_texts, model='text-embedding-3-small').data]
    col.add(ids=b_ids, documents=b_texts, embeddings=embs, metadatas=b_metas)
    bnum = bidx // BATCH + 1
    if bnum % 20 == 0 or bnum == total_batches:
        el = time.time() - t0
        rate = bnum / el
        eta = (total_batches - bnum) / rate if rate else 0
        print(f'  batch {bnum}/{total_batches} ({rate*BATCH:.0f} chunks/s, ETA {eta/60:.1f}m)')

print(f'\nChromaDB built: {col.count():,} chunks in {(time.time()-t0)/60:.1f}m')

## 6 · Pack outputs as zip (easier download)

In [ ]:
import shutil
zip_base = f'{OUTPUT_DIR}/rebuild_2026-05-26'
shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
sz = os.path.getsize(zip_base + '.zip') / 1024 / 1024
print(f'Zip: {zip_base}.zip ({sz:.0f} MB)')
print('\n=== DONE ===')
print('Tell Claude: "Colab rebuild done, files in Drive at vectorstore_rebuild_2026-05-26/"')